# 05 - Does graph consequence help? Engine validation + risk-controlled flagging
Runtime: **GPU** for the prediction cell (cell 6), CPU is fine for the rest if predictions are cached.

**Needs:** 02 v2 done, and the 4 **real-only fold runs** from 04 finished: `FOLD=0,1,2,3`, `USE_SYNTH=False`, `SEED=42` (exp ids `det_yolo11n_deeppcb_f{k}_s42_v2`). To test the synthetic-trained detectors instead, set `SYN='+synth1'` in cell 2 (needs those 4 runs too).

**Part A - consequence engine on real data.** For every annotated DeepPCB defect, compare template vs test copper graphs: does a net split (open) or two nets merge (short) at the box? Reports agreement with the annotated class. If this is weak, nothing downstream is worth building.

**Part B - decision experiment.** Out-of-fold detections from the 4 fold detectors (every board is scored by a detector that never saw it). Task: flag boxes so that at most a fraction alpha of truly *critical* defects (annotated open/short) is missed, with the threshold chosen by split-conformal calibration on held-out boards. Compare review load (boxes flagged per image) at the same alpha:
- **A0** raw detector confidence on critical classes
- **A** image-only learned score (conf, class, box size)
- **B** A + graph features (net split/merge at the box, agreement with predicted class, missing/excess copper)

Split per outer fold h: test = fold h boards, calibration = fold (h+1)%4, score-model fit = the other two folds. B is only interesting if it flags **fewer boxes than A at equal miss rate**.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os
D = '/content/drive/MyDrive/pcbrisk'
sys.path.insert(0, f'{D}/code')
from pcbrisk_utils import *
seed_all(42)

sh('pip -q install ultralytics scikit-image opencv-python-headless scipy scikit-learn pandas')
import numpy as np, pandas as pd, cv2, json, warnings
from scipy import ndimage as ndi
from skimage.morphology import remove_small_objects
from pathlib import Path
warnings.filterwarnings('ignore')

Mounted at /content/drive


In [2]:
SYN = ''             # '' = real-only detectors, '+synth1' = synthetic-trained detectors
SEED = 42
COPPER = 'auto'      # MUST match notebook 03 ('dark' | 'light' | 'auto')
MIN_AREA = 200       # ignore tiny nets
SIG = 20             # px overlap needed to count a piece/net as "the same"
PAD = 6              # box padding for the graph query
ALPHAS = [0.02, 0.05, 0.10]
CRIT = [0, 1]        # open, short
NAMES = ['open', 'short', 'mousebite', 'spur', 'copper', 'pin-hole']

In [3]:
restore_tar(f'{D}/01_prepared/prepared_v2.tar', f'{L}/data', marker='.ready_prepared_v2')
if not os.path.exists(f'{L}/raw/DeepPCB'):
    sh(f'unzip -q {D}/00_raw/deeppcb_main.zip -d {L}/raw')
mp = f'{D}/01_prepared/split_manifest_v2.csv'
assert sha256_file(mp) == open(f'{D}/01_prepared/manifest_v2.sha256').read(), 'manifest changed'
man = pd.read_csv(mp, dtype={'image_id': str})
P = Path(f'{L}/raw/DeepPCB/PCBData'); Y = Path(f'{L}/data/yolo_all')
temp_path = {p.name.replace('_temp.jpg', ''): p for p in P.glob('group*/*/*_temp.jpg')}
board_of = dict(zip(man.image_id, man.source_board)); fold_of = dict(zip(man.image_id, man.fold))

def load_gt(stem):
    rows = []
    for l in open(Y/'labels'/f'{stem}.txt'):
        v = l.split()
        if len(v) < 5: continue
        c, cx, cy, w, h = int(v[0]), *map(float, v[1:5])
        rows.append([c, (cx-w/2)*640, (cy-h/2)*640, (cx+w/2)*640, (cy+h/2)*640])
    return np.array(rows, float).reshape(-1, 5)
GT = {s: load_gt(s) for s in man.image_id}
print(len(GT), 'images,', sum(len(v) for v in GT.values()), 'GT boxes')

1500 images, 10013 GT boxes


In [4]:
def copper_mask(img, copper=COPPER):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, t = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = t > 0
    if copper == 'dark' or (copper == 'auto' and m.mean() > 0.5):
        m = ~m
    return remove_small_objects(m, 30)

S8 = np.ones((3, 3))
def analyse(stem):
    # graph comparison template vs test: which template nets split, which test components merge nets
    tp = cv2.imread(str(temp_path[stem])); xp = cv2.imread(str(Y/'images'/f'{stem}.jpg'))
    mt, mx = copper_mask(tp), copper_mask(xp)
    lt, nt = ndi.label(mt, structure=S8); lx, nx = ndi.label(mx, structure=S8)
    at = np.bincount(lt.ravel(), minlength=nt + 1); big = at >= MIN_AREA; big[0] = False
    pair = np.bincount(lt.ravel().astype(np.int64) * (nx + 1) + lx.ravel(), minlength=(nt + 1) * (nx + 1)).reshape(nt + 1, nx + 1)
    pair[:, 0] = 0; pair[0, :] = 0
    sig = pair >= SIG
    net_split = big & (sig.sum(1) >= 2)                      # a template net became >=2 pieces
    test_merged = (sig & big[:, None]).sum(0) >= 2           # a test component covers >=2 template nets
    return dict(mt=mt, mx=mx, lt=lt, lx=lx, net_split=net_split, test_merged=test_merged)

def query(a, box, pad=PAD):
    x1, y1, x2, y2 = [int(round(v)) for v in box]
    x1, y1, x2, y2 = max(0, x1-pad), max(0, y1-pad), min(640, x2+pad+1), min(640, y2+pad+1)
    mt, mx = a['mt'][y1:y2, x1:x2], a['mx'][y1:y2, x1:x2]
    missing, excess = mt & ~mx, mx & ~mt
    nets = np.unique(a['lt'][y1:y2, x1:x2][missing]); nets = nets[nets > 0]
    labs = np.unique(a['lx'][y1:y2, x1:x2][excess]); labs = labs[labs > 0]
    g_open = bool(a['net_split'][nets].any()) if len(nets) else False
    g_short = bool(a['test_merged'][labs].any()) if len(labs) else False
    return g_open, g_short, int(missing.sum()), int(excess.sum())

## Part A - does the graph engine agree with real annotations?

In [5]:
rows = []
for stem in man.image_id:
    g = GT[stem]
    if len(g) == 0: continue
    a = analyse(stem)
    for c, *bx in g:
        go, gs, ms, ex = query(a, bx)
        rows.append(dict(stem=stem, cls=int(c), g_open=go, g_short=gs, missing=ms, excess=ex))
va = pd.DataFrame(rows)
va['verdict'] = np.select([va.g_open & va.g_short, va.g_open, va.g_short], ['both', 'open', 'short'], 'none')
tab = pd.crosstab(va.cls.map(dict(enumerate(NAMES))), va.verdict, normalize='index').round(3)
print(tab.reindex(NAMES)); print()
print('annotated OPEN  -> engine says open :', round(((va.cls==0) & va.g_open).sum() / (va.cls==0).sum(), 3))
print('annotated SHORT -> engine says short:', round(((va.cls==1) & va.g_short).sum() / (va.cls==1).sum(), 3))
nc = va[~va.cls.isin(CRIT)]
print('non-critical classes wrongly flagged open/short:', round((nc.g_open | nc.g_short).mean(), 3))
va.to_csv(f'{D}/05_results/engine_validation.csv', index=False)
# READ THIS: high diagonal + low false-flag rate = engine is usable. Low diagonal: check COPPER polarity, SIG, PAD, MIN_AREA before going on.

verdict     both   none   open  short
cls                                  
open       0.231  0.111  0.560  0.098
short      0.297  0.056  0.122  0.526
mousebite  0.044  0.634  0.155  0.167
spur       0.048  0.604  0.153  0.195
copper     0.007  0.888  0.066  0.039
pin-hole   0.011  0.823  0.111  0.054

annotated OPEN  -> engine says open : 0.791
annotated SHORT -> engine says short: 0.823
non-critical classes wrongly flagged open/short: 0.273


## Part B - out-of-fold detections

In [6]:
from ultralytics import YOLO
pred_csv = f'{D}/05_results/oof_preds_{"real" if not SYN else SYN.strip("+")}_s{SEED}.csv'
if os.path.exists(pred_csv):
    preds = pd.read_csv(pred_csv, dtype={'stem': str}); print('loaded cached predictions', len(preds))
else:
    exps = {k: f'det_yolo11n_deeppcb{SYN}_f{k}_s{SEED}_v2' for k in range(4)}
    miss = [e for e in exps.values() if not os.path.exists(f'{D}/04_runs/{e}/weights/best.pt')]
    assert not miss, f'run these in notebook 04 first (FOLD, USE_SYNTH, SEED): {miss}'
    out = []
    for k, e in exps.items():
        mdl = YOLO(f'{D}/04_runs/{e}/weights/best.pt')
        stems = list(man[man.fold == k].image_id)
        paths = [str(Y/'images'/f'{s}.jpg') for s in stems]
        for r in mdl.predict(paths, conf=0.01, iou=0.6, imgsz=640, max_det=100, stream=True, verbose=False, batch=16):
            stem = Path(r.path).stem
            for (x1, y1, x2, y2), cf, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()):
                out.append((stem, k, int(c), float(cf), x1, y1, x2, y2))
        print('fold', k, 'done')
    preds = pd.DataFrame(out, columns=['stem', 'fold', 'cls', 'conf', 'x1', 'y1', 'x2', 'y2'])
    preds.to_csv(pred_csv, index=False)
print(preds.groupby('fold').size().to_dict(), 'boxes per fold')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
fold 0 done
fold 1 done
fold 2 done
fold 3 done
{0: 5551, 1: 7596, 2: 6373, 3: 3660} boxes per fold


In [7]:
# graph features for every predicted box
preds = preds.reset_index(drop=True)
feat = np.zeros((len(preds), 4));
for stem, sub in preds.groupby('stem'):
    a = analyse(stem)
    for i in sub.index:
        r = preds.loc[i]
        feat[i] = query(a, (r.x1, r.y1, r.x2, r.y2))
preds[['g_open', 'g_short', 'missing', 'excess']] = feat
preds['g_open'] = preds.g_open.astype(bool); preds['g_short'] = preds.g_short.astype(bool)
print(preds[['g_open', 'g_short']].mean().round(3).to_dict(), '<- share of low-conf boxes the engine calls open/short')

{'g_open': 0.233, 'g_short': 0.212} <- share of low-conf boxes the engine calls open/short


In [8]:
# targets: a predicted box is positive if it overlaps a CRITICAL ground-truth defect (IoU>=0.5), whatever class it claims
def iou_mat(a, b):
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    A = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); B = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (A[:, None] + B[None, :] - inter + 1e-9)

groups = {s: sub.index.values for s, sub in preds.groupby('stem')}
y = np.zeros(len(preds), int); gt_rows = []
for stem in man.image_id:
    g = GT[stem]; g = g[np.isin(g[:, 0], CRIT)]
    if len(g) == 0: continue
    idx = groups.get(stem, np.array([], int))
    I = iou_mat(preds.loc[idx, ['x1', 'y1', 'x2', 'y2']].values, g[:, 1:5]) if len(idx) else np.zeros((0, len(g)))
    if len(idx): y[idx] = (I.max(1) >= 0.5).astype(int)
    for j in range(len(g)):
        gt_rows.append(dict(stem=stem, fold=fold_of[stem], cls=int(g[j, 0]),
                            pred_idx=idx[I[:, j] >= 0.5] if len(idx) else np.array([], int)))
gt = pd.DataFrame(gt_rows); preds['y'] = y
print(len(gt), 'critical GT defects;', 'unmatched by any box:', int((gt.pred_idx.map(len) == 0).sum()), '| positive boxes:', int(y.sum()), 'of', len(preds))

3448 critical GT defects; unmatched by any box: 97 | positive boxes: 3883 of 23180


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

df = preds
df['crit'] = df.cls.isin(CRIT).astype(int)
df['conf_crit'] = df.conf * df.crit
df['logarea'] = np.log((df.x2 - df.x1).clip(1) * (df.y2 - df.y1).clip(1))
df['g_agree'] = (((df.cls == 0) & df.g_open) | ((df.cls == 1) & df.g_short)).astype(int)
df['g_cross'] = (((df.cls == 0) & df.g_short) | ((df.cls == 1) & df.g_open)).astype(int)
df['g_open'] = df.g_open.astype(int); df['g_short'] = df.g_short.astype(int)
df['lm'] = np.log1p(df.missing); df['le'] = np.log1p(df.excess); df['conf_agree'] = df.conf * df.g_agree

F_IMG = ['conf', 'crit', 'conf_crit', 'logarea']
F_GRAPH = F_IMG + ['g_open', 'g_short', 'g_agree', 'g_cross', 'lm', 'le', 'conf_agree']
POLICIES = {'A0 raw conf': None, 'A image-only': F_IMG, 'B +graph': F_GRAPH}

def gt_scores(sc, sub):
    return np.array([sc[i].max() if len(i) else -1.0 for i in sub.pred_idx])

def tau_for(s, alpha):                       # split-conformal quantile on critical-defect scores
    s = np.sort(s); k = int(np.floor(alpha * (len(s) + 1)))
    return -np.inf if k < 1 else s[k - 1]

res = []
for h in range(4):
    te, ca, fi = [h], [(h + 1) % 4], [(h + 2) % 4, (h + 3) % 4]
    for pol, feats in POLICIES.items():
        if feats is None:
            sc = (df.conf * df.crit).values
        else:
            mdl = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
            m_fit = df.fold.isin(fi)
            mdl.fit(df.loc[m_fit, feats], df.loc[m_fit, 'y']); sc = mdl.predict_proba(df[feats])[:, 1]
        s_cal = gt_scores(sc, gt[gt.fold.isin(ca)]); s_te = gt_scores(sc, gt[gt.fold.isin(te)])
        mte = (df.fold == h).values; n_img = (man.fold == h).sum()
        for al in ALPHAS:
            tau = tau_for(s_cal, al)
            miss = ((s_te < tau) | (s_te == -1)).sum()
            flag = mte & (sc >= tau)
            res.append(dict(policy=pol, alpha=al, fold=h, n_gt=len(s_te), n_miss=int(miss), tau=tau,
                            flagged_per_img=flag.sum() / n_img, precision=df.y[flag].mean() if flag.any() else np.nan))
R = pd.DataFrame(res)
R.to_csv(f'{D}/05_results/flagging_results_{"real" if not SYN else SYN.strip("+")}_s{SEED}.csv', index=False)

In [10]:
summ = R.groupby(['alpha', 'policy']).apply(lambda g: pd.Series(dict(
    miss_rate=g.n_miss.sum() / g.n_gt.sum(),
    flagged_per_img=g.flagged_per_img.mean(), flagged_std=g.flagged_per_img.std(),
    precision=g.precision.mean()))).round(3)
print(summ)
print('\nB vs A (image-only), flagged boxes per image at the same alpha:')
for al in ALPHAS:
    a_ = R[(R.alpha == al) & (R.policy == 'A image-only')].set_index('fold').flagged_per_img
    b_ = R[(R.alpha == al) & (R.policy == 'B +graph')].set_index('fold').flagged_per_img
    d = (b_ - a_)
    print(f'alpha={al}: B-A = {d.mean():+.3f} +- {d.std():.3f} per image | folds where B flags fewer: {(d < 0).sum()}/4')

                    miss_rate  flagged_per_img  flagged_std  precision
alpha policy                                                          
0.02  A image-only      0.043           10.032        7.784      0.421
      A0 raw conf       0.042            9.816        7.904      0.429
      B +graph          0.039            9.789        7.869      0.417
0.05  A image-only      0.090            4.191        1.978      0.640
      A0 raw conf       0.087            5.792        4.004      0.571
      B +graph          0.070            3.939        1.701      0.677
0.10  A image-only      0.178            2.195        0.511      0.884
      A0 raw conf       0.178            2.185        0.531      0.884
      B +graph          0.137            2.189        0.369      0.911

B vs A (image-only), flagged boxes per image at the same alpha:
alpha=0.02: B-A = -0.243 +- 0.883 per image | folds where B flags fewer: 1/4
alpha=0.05: B-A = -0.251 +- 0.826 per image | folds where B flags fewer: 2/4


## How to read this
- **Part A** is the engine's credibility. Report the agreement and false-flag rates as they are.
- **Part B** supports the idea only if B (graph) flags clearly fewer boxes than A at the same alpha **and** its miss rate stays near alpha, in most folds. If B is within noise of A, the graph features add nothing on this data. That is a legitimate result: drop the claim or change the problem.
- Miss rate can exceed alpha: the conformal guarantee assumes calibration and test boards are exchangeable, and boards are not. Report it, do not hide it.
- If an alpha is infeasible (many critical defects have no box at all), `tau=-inf` and everything is flagged. That is a detector recall limit, not a calibration bug.
- DeepPCB defects are partly artificial and the nets are raster components, so this is a proof of mechanism, not evidence about production boards.
- Repeat with `SEED=43` (after running those fold models in 04) and `SYN='+synth1'` for more evidence.